In [12]:
import sys
from pathlib import Path
import os

print("Current working directory:", os.getcwd())

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

print("Project root added:", PROJECT_ROOT)
print("database.py exists:", (PROJECT_ROOT / "database.py").exists())

Current working directory: c:\Users\jobs\Desktop\Suleman\code\employee_onboarding\notebooks
Project root added: c:\Users\jobs\Desktop\Suleman\code\employee_onboarding
database.py exists: True


In [13]:
from database import create_tables, SessionLocal, Employee

print("Import successful")


Import successful


In [17]:
create_tables()
print("Tables created successfully")

2026-03-19 15:18:18,147 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-19 15:18:18,148 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("employees")
2026-03-19 15:18:18,149 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-19 15:18:18,150 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("employees")
2026-03-19 15:18:18,150 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-19 15:18:18,151 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("audit_log")
2026-03-19 15:18:18,152 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-19 15:18:18,152 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("audit_log")
2026-03-19 15:18:18,152 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-19 15:18:18,155 INFO sqlalchemy.engine.Engine 
CREATE TABLE employees (
	id INTEGER NOT NULL, 
	first_name VARCHAR NOT NULL, 
	last_name VARCHAR NOT NULL, 
	gender VARCHAR NOT NULL, 
	date_of_birth VARCHAR NOT NULL, 
	country_of_birth VARCHAR NOT NULL, 
	place_of_birth VARCHAR NOT NULL, 
	nat

In [ ]:
db = SessionLocal()

emp = Employee(
    first_name="Suleman",
    last_name="Test",
    gender="Male",
    date_of_birth="10.10.1995",
    country_of_birth="Pakistan",
    place_of_birth="Lahore",
    nationality="Pakistani",
    address="123 Main St",
    zip_code="12345",
    email="suleman@test.com",
    steuer_id="123456789",
    steuerklasse=1,
    iban="DE89370400440532013000",
    start_date="2023-01-01",
    contract_type="Full-time",
    weekly_hours=40,
    hourly_rate="50.00",
)

db.add(emp)
db.commit()
db.refresh(emp)

print("Inserted employee:", emp.id, emp.first_name, emp.status)

db.close()


TypeError: 'phone' is an invalid keyword argument for Employee

In [11]:
from datetime import datetime
from pathlib import Path

from sqlalchemy import create_engine, Column, Integer, String, DateTime, Boolean
from sqlalchemy.orm import declarative_base, sessionmaker


# Absolute project paths based on the current working directory (Jupyter notebooks don't define __file__)
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

DATABASE_PATH = DATA_DIR / "employees.db"
DATABASE_URL = f"sqlite:///{DATABASE_PATH}"


engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

Base = declarative_base()


class Employee(Base):
    __tablename__ = "employees"

    id = Column(Integer, primary_key=True, index=True)
    status = Column(String, default="draft")
    created_at = Column(DateTime, default=datetime.utcnow)
    updated_at = Column(DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)
    is_deleted = Column(Boolean, default=False)

    first_name = Column(String, nullable=True)
    last_name = Column(String, nullable=True)
    date_of_birth = Column(String, nullable=True)
    place_of_birth = Column(String, nullable=True)

    phone = Column(String, nullable=True)
    email = Column(String, nullable=True)

    street_and_number = Column(String, nullable=True)
    zip_code = Column(String, nullable=True)
    city = Column(String, nullable=True)
    country = Column(String, nullable=True, default="Deutschland")


def create_tables():
    Base.metadata.create_all(bind=engine)


def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()